# SEED-VII EEGNet x LoRA-LLM — Kaggle Pipeline

跨 Session 可续训。代码和所有中间产物都以 Kaggle Dataset 持久化——
新 Session 启动时从 Kaggle 拉回，无需重算、无需 git clone。

| Kaggle Dataset | 内容 |
|---------------|------|
| `seed-vii-eeg` | 1-20.mat + text_protocol.csv + 仓库代码 |
| `seed-vii-npz` | index.csv + shard_*.npz + norm_stats.npz |
| `seed-vii-llm` | Qwen2.5-0.5B-Instruct |
| `seed-vii-runs` | checkpoints + config.yaml + embeddings |

## 0. 环境 + 配置（全环境变量，零交互）

In [ ]:
import sys, os, shutil, time, re
from pathlib import Path

os.environ.setdefault('MODELSCOPE_TOKEN', 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2')
os.environ.setdefault('KAGGLE_USERNAME', 'PRIMOCOSMOS')
os.environ.setdefault('DATASET_ID', 'DEREKVERSE/SEED-VII')
os.environ['GIT_TERMINAL_PROMPT'] = '0'

MODELSCOPE_TOKEN = os.environ['MODELSCOPE_TOKEN']
KAGGLE_USERNAME  = os.environ['KAGGLE_USERNAME']
DATASET_ID       = os.environ['DATASET_ID']

# 四个 Kaggle Dataset handles — 代码在 seed-vii-eeg 里
KAGGLE_EEG  = os.environ.get('KAGGLE_EEG',  f'{KAGGLE_USERNAME}/seed-vii-eeg')
KAGGLE_NPZ  = os.environ.get('KAGGLE_NPZ',  f'{KAGGLE_USERNAME}/seed-vii-npz')
KAGGLE_LLM  = os.environ.get('KAGGLE_LLM',  f'{KAGGLE_USERNAME}/seed-vii-llm')
KAGGLE_RUNS = os.environ.get('KAGGLE_RUNS', f'{KAGGLE_USERNAME}/seed-vii-runs')

WORK = Path('/kaggle/working'); TMP = Path('/kaggle/tmp')

for d in [WORK, TMP]: d.mkdir(parents=True, exist_ok=True)
for l, d in [('/kaggle/working', WORK), ('/kaggle/tmp', TMP)]:
    du = shutil.disk_usage(d)
    print(f'[DISK] {l}: {du.free/(1024**3):.1f} GB free')

%pip install -q modelscope kagglehub h5py 2>&1 | tail -1
print('[OK]')

## 1. 加载各 Kaggle Dataset + 定位代码仓库

In [ ]:
import kagglehub
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
from seedvii_contrastive.data.discovery import SUBJECT_FILE_NAMES

def _kaggle_load(handle, label=''):
    """加载 Kaggle Dataset，返回 Path 或 None。自动探测单子目录包装。"""
    print(f'[{label}] {handle}')
    try:
        dl = Path(kagglehub.dataset_download(
            handle=handle, output_dir=str(WORK), force_download=False
        )).resolve()
        items = list(dl.iterdir())
        files = [x for x in items if x.is_file()]
        dirs  = [x for x in items if x.is_dir()]
        if not files and len(dirs) == 1:
            dl = dirs[0]
        print(f'[{label}] -> {dl}')
        return dl
    except Exception as e:
        print(f'[{label}] {e}')
        return None

def _kaggle_upload(handle, src_dir, note, label=''):
    n = len(list(Path(src_dir).iterdir()))
    print(f'[{label}] Upload {n} entries -> {handle}')
    for a in range(1, 4):
        try:
            kagglehub.dataset_upload(handle=handle, local_dataset_dir=str(src_dir),
                                      version_notes=note)
            print(f'[{label}] Done')
            return
        except Exception as e:
            print(f'[{label}] Fail {a}/3: {e}')
            if a < 3: time.sleep(30 * a)
            else: raise

# ---- 1a. EEG 数据集（含代码仓库）----
EEG_PATH  = _kaggle_load(KAGGLE_EEG, 'EEG')
EEG_READY = False
PROJ = None

if EEG_PATH:
    # 先找 seedvii_modal_contrastive_lora（可能在 EEG_PATH 的子目录里）
    for candidate in [
        EEG_PATH / 'seedvii_modal_contrastive_lora',
        EEG_PATH / 'EEG_OPUS1' / 'seedvii_modal_contrastive_lora',
        EEG_PATH / 'EEG_OPUS' / 'seedvii_modal_contrastive_lora',
    ]:
        if (candidate / 'pyproject.toml').exists():
            PROJ = candidate
            break
    # 如果上面没找到，全局搜索
    if PROJ is None:
        hits = list(EEG_PATH.rglob('seedvii_modal_contrastive_lora/pyproject.toml'))
        if hits:
            PROJ = hits[0].parent

    # 找 EEG 数据文件
    er, tc = find_downloaded_paths(EEG_PATH)
    if er:
        mats = sorted(p.name for p in Path(er).glob('*.mat') if p.name in SUBJECT_FILE_NAMES)
        if len(mats) >= 20 and tc:
            EEG_READY = True
            EEG_ROOT, TEXT_CSV = er, tc
            print(f'[EEG] {len(mats)} .mat + CSV')

# ---- 1b. NPZ ----
NPZ_PATH  = _kaggle_load(KAGGLE_NPZ, 'NPZ')
NPZ_READY = bool(NPZ_PATH and (NPZ_PATH / 'index.csv').exists())

# ---- 1c. LLM ----
LLM_PATH  = _kaggle_load(KAGGLE_LLM, 'LLM')
LLM_READY = bool(LLM_PATH and (LLM_PATH / 'config.json').exists())

# ---- 1d. Runs ----
RUNS_PATH = _kaggle_load(KAGGLE_RUNS, 'RUNS')
RUNS_READY = bool(RUNS_PATH and list(RUNS_PATH.iterdir()))
if RUNS_READY:
    print('[RUNS] Will resume from checkpoint')

# ---- 安装依赖 + 导入路径 ----
if PROJ is not None:
    print(f'[PROJ] Found in dataset: {PROJ}')
    %pip install -q -r {PROJ / 'requirements.txt'} 2>&1 | tail -1
    %pip install -q -e {PROJ} 2>&1 | tail -1
else:
    # 回退：git clone
    print('[PROJ] Not in dataset, cloning from GitHub ...')
    REPO = WORK / 'EEG_OPUS1'
    !git clone --depth 1 https://github.com/PRIMOCOSMOS/EEG_OPUS.git {REPO} 2>&1 | tail -1
    PROJ = REPO / 'seedvii_modal_contrastive_lora'
    %pip install -q -r {PROJ / 'requirements.txt'} 2>&1 | tail -1
    %pip install -q -e {PROJ} 2>&1 | tail -1

if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))

# 重新 import（确保使用正确的项目路径）
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
from seedvii_contrastive.data.discovery import SUBJECT_FILE_NAMES

print(f'[READY] PROJ={PROJ}  EEG={EEG_READY}  NPZ={NPZ_READY}  LLM={LLM_READY}  RUNS={RUNS_READY}')

## 2. EEG — 没有就从 ModelScope 下载

In [ ]:
if not EEG_READY:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import threading
    from modelscope.hub.api import HubApi
    from modelscope.hub.file_download import dataset_file_download

    MW = int(os.environ.get('MAX_WORKERS', '8'))
    lk = threading.Lock()

    print(f'[LIST] {DATASET_ID}')
    api = HubApi(token=MODELSCOPE_TOKEN)
    af, pg = [], 1
    while True:
        b = api.get_dataset_files(repo_id=DATASET_ID, revision='master',
                                  root_path='/', recursive=True,
                                  page_number=pg, page_size=200)
        if not b: break
        af.extend(b); pg += 1
        if len(b) < 200: break

    tg, sz, gb = {}, {}, 0
    for f in af:
        p = f.get('Path') or f.get('Name') or ''; n = Path(p).name
        if re.match(r'^([1-2][0-9]|[1-9])\.mat$', n):
            tg[n] = p; sz[n] = int(f.get('Size', 0)); gb += sz[n] / 1e9
        elif n == 'text_protocol.csv':
            tg[n] = p; sz[n] = int(f.get('Size', 0))
    print(f'[FIND] {len(tg)} files, ~{gb:.1f} GB')

    du = shutil.disk_usage(TMP)
    if du.free / 1e9 < gb * 1.2:
        raise RuntimeError(f'/kaggle/tmp: {du.free/1e9:.1f} < {gb*1.2:.1f} GB')
    assert all(f'{i}.mat' in tg for i in range(1, 21)) and 'text_protocol.csv' in tg

    SG = TMP / 'seedvii_upload'
    SG.mkdir(parents=True, exist_ok=True)

    def dl_one(n, rp, dd, es, mt=5):
        d = dd / n; mb = int(es * 0.9)
        for a in range(1, mt + 1):
            try:
                p = Path(dataset_file_download(dataset_id=DATASET_ID, file_path=rp,
                       revision='master', local_dir=str(dd), token=MODELSCOPE_TOKEN))
                if p != d: d.unlink(missing_ok=True); shutil.copy2(str(p), str(d)); p.unlink(missing_ok=True)
                if d.stat().st_size < mb: d.unlink(missing_ok=True); raise RuntimeError('size')
                with lk: print(f'  [OK] {n:12s} ({d.stat().st_size/1e6:7.1f} MB)')
                return (n, d, None)
            except Exception as e:
                d.unlink(missing_ok=True)
                if a < mt:
                    w = min(2**a, 60)
                    with lk: print(f'  [RETRY {a}/{mt}] {n}: {e} - {w}s')
                    time.sleep(w)
                else:
                    with lk: print(f'  [FAIL] {n}: {e}')
                    return (n, None, str(e))
        return (n, None, 'unknown')

    td = []
    for n, rp in sorted(tg.items()):
        d = SG / n; ms = int(sz.get(n, 0) * 0.9)
        if d.exists() and d.stat().st_size >= ms: print(f'  [SKIP] {n}'); continue
        d.unlink(missing_ok=True); td.append((n, rp))

    if td:
        print(f'\n[DL] {len(td)} files, {MW} workers')
        t0 = time.time(); fl = []
        with ThreadPoolExecutor(max_workers=min(MW, len(td))) as pool:
            fs = {pool.submit(dl_one, n, r, SG, sz[n]): n for n, r in td}
            for f in as_completed(fs): n, _, e = f.result(); (fl.append(n) if e else None)
        et = time.time() - t0; ok = len(td) - len(fl)
        print(f'\n[DL] {ok}/{len(td)} OK ({et:.0f}s)')
        if fl: raise RuntimeError(f'Failed: {fl}')

    _kaggle_upload(KAGGLE_EEG, SG, f'{len(tg)} files (MS {DATASET_ID})', 'EEG')
    shutil.rmtree(SG, ignore_errors=True)

    EEG_PATH = _kaggle_load(KAGGLE_EEG, 'EEG')
    er, tc = find_downloaded_paths(EEG_PATH)
    assert er and tc
    EEG_ROOT, TEXT_CSV = er, tc
    EEG_READY = True
else:
    print('[EEG] Skipped')

## 3. NPZ — 没有就预处理

In [ ]:
if not NPZ_READY:
    assert EEG_READY
    NPZ_PATH = WORK / 'seedvii_npz'
    NPZ_PATH.mkdir(parents=True, exist_ok=True)
    print('[NPZ] Preprocessing ...')
    !python -m seedvii_contrastive.scripts.preprocess_npz \
        --input-root {EEG_ROOT} --output-dir {NPZ_PATH} \
        --subjects 1-20 --window-sec 4 --stride-sec 4 \
        --center-ratio 0.60 --max-windows-per-clip 12 --shard-size 512
    _kaggle_upload(KAGGLE_NPZ, NPZ_PATH, 'NPZ shards', 'NPZ')
    NPZ_READY = True
else:
    print('[NPZ] Skipped')

## 4. LLM — 没有就下载

In [ ]:
if not LLM_READY:
    LLM_PATH = WORK / 'models' / 'Qwen2.5-0.5B-Instruct'
    LLM_PATH.mkdir(parents=True, exist_ok=True)
    print('[LLM] Downloading Qwen2.5-0.5B-Instruct ...')
    from modelscope import snapshot_download
    mp = snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', cache_dir=str(LLM_PATH.parent))
    mp = Path(mp)
    if mp.resolve() != LLM_PATH.resolve():
        if LLM_PATH.exists(): shutil.rmtree(LLM_PATH)
        shutil.copytree(str(mp), str(LLM_PATH), symlinks=True)
    _kaggle_upload(KAGGLE_LLM, LLM_PATH, 'Qwen2.5-0.5B-Instruct', 'LLM')
    LLM_READY = True
else:
    print('[LLM] Skipped')

## 5. Runs — 没有就创建

In [ ]:
if not RUNS_READY:
    RUNS_PATH = WORK / 'runs' / 'run_valence3'
    RUNS_PATH.mkdir(parents=True, exist_ok=True)
    print('[RUNS] Starting fresh')
else:
    print('[RUNS] Resuming')

## 6. 训练

In [ ]:
assert EEG_READY and NPZ_READY and LLM_READY
assert EEG_ROOT and TEXT_CSV
RUNS_PATH.mkdir(parents=True, exist_ok=True)

import yaml
cfg = yaml.safe_load(open(PROJ / 'configs' / 'modelscope_default.yaml'))
cfg['data'].update({k: str(v) for k, v in dict(
    modelscope_dataset_id=DATASET_ID,
    local_dataset_dir=str(EEG_PATH),
    eeg_root=str(EEG_ROOT),
    text_csv_path=str(TEXT_CSV),
    npz_dir=str(NPZ_PATH),
).items()})
cfg['runtime']['output_dir'] = str(RUNS_PATH)
cfg['model']['llm']['model_name_or_path'] = str(LLM_PATH)
cfg['model']['llm']['gradient_checkpointing'] = True
cfg['train']['batch_size'] = 48

rc = RUNS_PATH / 'config.yaml'
yaml.safe_dump(cfg, open(rc, 'w'), allow_unicode=True, sort_keys=False)
!python -m seedvii_contrastive.scripts.train_contrastive --config {rc}

## 7. 保存 Runs → Kaggle（下个 Session 续训）

In [ ]:
_kaggle_upload(KAGGLE_RUNS, RUNS_PATH,
                f'Checkpoint {time.strftime("%Y-%m-%d %H:%M")}', 'RUNS')

In [ ]:
BEST = RUNS_PATH / 'best.pt'
if BEST.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg --config {rc} --checkpoint {BEST} --split val --out {RUNS_PATH/'emb.npz'}
else:
    print('[WARN] No best.pt')